##### Copyright 2025 Google LLC。

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# 使用 Gemma 和 Gradio 建立聊天機器人

<table align="left"> <td>    <a target="_blank" href="https://colab.research.google.com/github/google-gemma/cookbook/blob/main/.archive/Gemma/[Gemma_2]Gradio_Chatbot.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
</td>
</table>

## 設定

### 運作環境

  1. 按一下「**在 Colab** 中開啟」。
  2. In the menu, go to **Runtime** > **Change runtime type**.
  3. 在 **硬體加速器** 下，選擇 **T4 GPU**。


### Hugging Face Hub 訪問 token

在深入學習本教學之前，讓我們先設定Gemma：
1. **建立一個 Hugging Face 帳戶**：如果您沒有帳戶，您可以[此處]註冊一個免費帳戶(https://huggingface.com/join)。
2. **造訪Gemma模型**：造訪[Gemma模型頁面](https://huggingface.com/collections/google/gemma-2-release-667d6600fd5220e7b967f315)並接受使用條款。
3. **產生Hugging Facetoken**：前往您的Hugging Face [設定頁面](https://huggingface.com/settings/tokens)並產生新的存取token（最好具有`write`權限）。在本教學的後面部分，您將需要這個token。

**完成這些步驟後，您就可以進入下一部分，在 Colab 環境中設定環境變數。 **

### 設定您的憑證

要存取私有模型和dataset，您需要登入Hugging Face（HF）生態系統。
如果您使用Colab，您可以使用Colab Secrets Manager 安全地儲存您的Hugging Face token (`HF_TOKEN`)：1. 開啟 Google Colab notebook 並點選左側面板中的 🔑 Secrets 標籤。 <img src="https://storage.googleapis.com/generativeai-downloads/images/secrets.jpg" alt="The Secrets tab is found on the left panel." width=50%>
2. **新增Hugging Facetoken**：
- 建立一個新的secret，其**名稱**為`HF_TOKEN`。
- Copy and paste your token key into the **Value** input box for `HF_TOKEN`.
- **Toggle** the button on the left to allow notebook access to the secret

此程式碼會擷取您的 secrets 並將它們設定為環境變數以供本教學後面使用。

In [ ]:
import os
import sys

if "google.colab" in sys.modules:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get("HF_TOKEN")

if "HF_TOKEN" not in os.environ:
    raise EnvironmentError(
        "The Hugging Face token (HF_TOKEN) could not be found in the "
        "environment variables. This token is required to download the Gemma "
        "models from the Hugging Face Hub. For more information about "
        "HF User Access tokens, please refer to the HF documentation "
        "here: https://huggingface.co/docs/hub/en/security-tokens."
    )

### 安裝依賴項

接下來，您將安裝所需的 library。在這種情況下，我們只需要聊天介面的 gradio 和變壓器來從 Hugging Face Hub 載入 Gemma 模型。

In [ ]:
!pip install -q -U gradio==5.9.1
!pip install -q -U transformers==4.46.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.4/320.4 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 57.1 MB/s eta 0:00:00


## 使用Gradio 與Gemma 聊天

### 正在初始化Gemma 2模型

讓我們建立一個使用 gemma-2-2b-it 模型產生文字的管道。變壓器 library 提供了一種簡單的方法，只需指定模型名稱和一些基本參數即可將模型和 tokenizer 載入到記憶體中。

In [ ]:
import torch
import transformers

# Model details
model_name = "google/gemma-2-2b-it"
device = "cuda"
model_kwargs = {
    "torch_dtype": torch.float16,
}

# Load the Gemma tokenizer
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)

# Create a pipeline
pipeline = transformers.pipeline(
    "text-generation",
    model=model_name,
    device=device,
    model_kwargs=model_kwargs
)

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/241M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

### 建立自訂聊天模板

Hugging Face 支援聊天模板，該模板可以定義將對話轉換為單一 token 可化字串的結構和格式，這是語言模型期望的輸入格式。查看[聊天範本文件](https://huggingface.co/docs/transformers/main/en/chat_templating) 以了解有關範本以及如何建立自訂範本的更多資訊。
由於Gemma不支援系統指令，因此您將提供系統輸入作為使用者輸入。本範本已調整為相容Gradio的聊天介面。要了解有關 Gemma 所需格式的更多信息，請查看 [Gemma 格式文件](https://ai.google.dev/gemma/docs/formatting)。

In [ ]:
tokenizer.chat_template = \
    "{{ bos_token }}"\
    "{% if messages[0]['role'] == 'system' %}"\
        "{{'<start_of_turn>user\n' + messages[0]['content'] | trim + ' ' + messages[1]['content'] | trim + '<end_of_turn>\n'}}"\
        "{% set messages = messages[2:] %}"\
    "{% endif %}"\
    "{% for message in messages %}"\
        "{% if message['role'] == 'user' %}"\
            "{{'<start_of_turn>user\n' + message['content'] | trim + '<end_of_turn>\n'}}"\
        "{% elif message['role'] == 'assistant' %}"\
            "{{'<start_of_turn>model\n' + message['content'] | trim + '<end_of_turn>\n' }}"\
        "{% endif %}"\
    "{% endfor %}"\
    "{% if add_generation_prompt %}"\
        "{{ '<start_of_turn>model\n' }}"\
    "{% endif %}"

### 處理新訊息

現在，您需要定義一個處理新訊息（使用者輸入）的函數。
為了使模型能夠感知上下文，我們需要提供：
1. 系統訊息：對話的第一個訊息，指導模型在聊天過程中的行為。
1. 聊天記錄：到目前為止助手與用戶之間交換的訊息。
1. 新訊息：用戶發送的新訊息。

所有這些資訊都會轉換為訊息清單。然後，`apply_chat_template`用於建立實際的prompt（長字串，其中包含Gemma所需的所有特殊tokens）。 prompt 傳遞到tokenizer，然後傳遞到模型以產生回應。

In [ ]:
from typing import List, Dict

system_message = "You're a helpful assistant."

def chat_with_gemma(message: str, history: List[Dict[str, str]],
                    max_new_tokens: int = 512) -> str:
    """Chats with the Gemma 2 model and returns the response.

    This function takes a user message and chat history as input, formats them
    using the custom chat template, and generates a response using the Gemma 2
    pipeline.

    Args:
        message:        The user's message.
        history:        The chat history as a list of messages.
        max_new_tokens: The maximum number of new tokens to generate.

    Returns:
        response: Content generated by the model.
    """

    # Combine system message, history and the new message into a list of messages.
    messages = [
        {"role": "system", "content": system_message},
        *history,
        {"role": "user", "content": message},
    ]

    # Apply the chat template to convert it into the prompt (string).
    prompt = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )

    # Generate response using the pipeline defined above.
    outputs = pipeline(prompt, max_new_tokens=max_new_tokens)

    # A basic error handling mechanism. If something goes wrong, the
    # user will see "Something went wrong..." instead of a long error message.
    # It's usually a good place to handle quota limits, harmful content, etc.
    response = "_Something went wrong. Please try again._"
    try:
        response = outputs[0]["generated_text"][len(prompt):]
    except:
        pass
    return response

### 讓我們執行吧！

現在，我們將使用Gradio的`ChatInterface`來建立互動式聊天介面，讓您可以與我們的Gemma 2模型聊天！在本例中，它將在 Google Colab 內建立一個窗口，但如果您在獨立檔案中執行它，它將啟動一個 HTTP 伺服器，您將能夠從瀏覽器存取聊天。

In [ ]:
import gradio as gr

gr.ChatInterface(
    fn=chat_with_gemma,
    type="messages"
).launch()

Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://29b77c0650271c6a24.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 接下來是什麼？

就是這樣！如果您想知道如何讓您的 chatbot 變得更好，請查看以下資源：
- **探索 Gemma 系列型號：** 造訪 [Gemma 開放型號](https://ai.google.dev/gemma) 了解 Gemma 系列型號的最新更新、新功能、版本等。
- **Gradio 自訂：** 瀏覽 [Gradio 文件](https://www.gradio.app/docs) 以了解如何自訂聊天介面、新增選項和功能。
- **分享您的Gradio儀表板：**查看[共享您的Gradio應用程式](https://www.gradio.app/guides/sharing-your-app)頁面，了解如何與其他人安全地共享您的Gradio儀表板！